# Misure esterne dei paesi, anno per anno

Raccoglie in un'unica tabella per anno le misure macro dei paesi che servono per l'econometria. Non dipendono dalla risoluzione dei prodotti.

| colonna | descrizione | fonte |
|---|---|---|
| `rgdpna`, `rgdpe` | PIL reale (prezzi nazionali costanti / PPP lato spesa), mln $ 2021 | Penn World Table 11.0 |
| `pop`, `emp` | popolazione, occupati (milioni) | PWT |
| `hc` | indice di capitale umano | PWT |
| `lny` | log PIL pro capite reale = log(`rgdpna` / `pop`) | PWT |
| `gdp_usd`, `gdppc_usd` | PIL e PIL pro capite, $ correnti | World Bank WDI |
| `gdppc_growth` | crescita annua PIL pro capite (%, prezzi costanti) | World Bank WDI |
| `hdi`, `le`, `eys`, `mys`, `gnipc` | HDI e componenti | UNDP HDR 2025 |

Solo codici ISO3 presenti nei codici paese BACI (niente aggregati regionali).

Output: `results/econometria/misure_esterne/country/Y{year}.parquet` con `year, iso3, <misure>` (vedi `src/storage.py`).

In [1]:
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path().resolve(), *Path().resolve().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

from src.storage import RAW_DIR, YEARS, external_country_path

## Dati

In [2]:
countries = set(pd.read_csv(ROOT / "data" / "raw" / "BACI" / "country_codes_V202601.csv")["country_iso3"].dropna())

pwt = pd.read_excel(RAW_DIR / "pwt110.xlsx", sheet_name="Data",
                    usecols=["countrycode", "year", "rgdpna", "rgdpe", "pop", "emp", "hc"])
pwt = pwt.rename(columns={"countrycode": "iso3"}).set_index(["iso3", "year"])
pwt["lny"] = np.log(pwt["rgdpna"] / pwt["pop"])

WDI = {"gdp_usd": "API_NY.GDP.MKTP.CD_DS2_en_csv_v2_358370",
       "gdppc_usd": "API_NY.GDP.PCAP.CD_DS2_en_csv_v2_353836",
       "gdppc_growth": "API_NY.GDP.PCAP.KD.ZG_DS2_en_csv_v2_346194"}

def wdi(name):
    with zipfile.ZipFile(RAW_DIR / f"{name}.zip") as z:
        d = pd.read_csv(z.open(f"{name}.csv"), skiprows=4)
    d = d.rename(columns={"Country Code": "iso3"}).set_index("iso3")
    d = d[[c for c in d.columns if c.isdigit()]]
    s = d.stack().rename_axis(["iso3", "year"])
    s.index = s.index.set_levels(s.index.levels[1].astype(int), level="year")
    return s

wb = pd.concat({k: wdi(v) for k, v in WDI.items()}, axis=1)

hdr = pd.read_excel(RAW_DIR / "hdr-data.xlsx")
hdr = hdr[hdr["indicatorCode"].isin(["hdi", "le", "eys", "mys", "gnipc"])]
hdr = hdr.pivot_table(index=["countryIsoCode", "year"], columns="indicatorCode", values="value")
hdr.index.names = ["iso3", "year"]; hdr.columns.name = None

ext = pwt.join(wb, how="outer").join(hdr, how="outer").reset_index()
ext = ext[ext["iso3"].isin(countries) & ext["year"].isin(YEARS)]
ext.groupby("year").count().drop(columns="iso3")

,rgdpe,pop,emp,hc,rgdpna,lny,gdp_usd,gdppc_usd,gdppc_growth,eys,gnipc,hdi,le,mys
year,,,,,,,,,,,,,,
2002,181,181,179,144,181,181,202,202,196,184,189,171,193,179
2003,181,181,178,144,181,181,202,202,199,184,189,172,193,180
2004,181,181,178,144,181,181,202,202,199,184,189,174,193,182
2005,183,183,178,144,183,183,202,202,199,189,190,185,193,188
2006,183,183,178,144,183,183,203,203,199,189,190,185,193,188
2007,183,183,178,144,183,183,203,203,200,190,190,186,193,188
2008,184,184,179,144,184,184,204,204,201,190,192,187,193,189
2009,184,184,177,144,184,184,205,205,202,190,192,186,193,189
2010,184,184,178,144,184,184,205,205,203,192,192,190,193,190


## Salvataggio

In [3]:
for year, g in ext.groupby("year"):
    path = external_country_path(year)
    path.parent.mkdir(parents=True, exist_ok=True)
    g.sort_values("iso3").reset_index(drop=True).to_parquet(path)
print(f"salvati {ext.year.nunique()} anni in {path.parent}")

salvati 23 anni in C:\Users\vitto\Desktop\CSH RESEARCH\results\econometria\misure_esterne\country
